# Labelling NISAR chips

Runs in **Google Colab** or on your **own machine** — the notebook detects which and
adapts. Partners without the granule use Colab; if you already have the `.h5` locally,
run it locally and skip the uploads.

**One thing cannot be automated: QGIS does not run in Colab.** It is a desktop program
and Colab is a headless Linux box. So the work splits:

| Where | What happens |
|---|---|
| This notebook | cut your chips, create your annotation files, export the final masks |
| QGIS, on your laptop | draw the polygons |

Everything here just calls the `nisar-flood` command line, so you can run the same
commands in a terminal if you would rather.


## 1. Set up

In Colab this clones the repo and installs it (about two minutes, once per session).
Locally it does nothing except check you are in the right directory.


In [ ]:
import os, subprocess, sys

try:
    import google.colab           # present only on Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
REPO = 'https://github.com/cs59300-deep-learning/nisar-flood-mapping.git'
BRANCH = 'feat/28-labelling-tooling'   # change to 'main' once PR #60 merges

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'rasterio', 'h5py'], check=True)
    if not os.path.exists('/content/nisar-flood-mapping'):
        subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, REPO,
                        '/content/nisar-flood-mapping'], check=True)
    os.chdir('/content/nisar-flood-mapping')

assert os.path.exists('pyproject.toml'), (
    f'run this from the repository root; cwd is {os.getcwd()}')


def nisar(*arguments):
    """Run a nisar-flood subcommand and show its output.

    Invoking the CLI as a module with src on the path means the notebook
    works whether or not the package has been pip-installed.
    """
    environment = dict(os.environ)
    environment['PYTHONPATH'] = os.pathsep.join(
        ['src', environment.get('PYTHONPATH', '')]).rstrip(os.pathsep)
    finished = subprocess.run([sys.executable, '-m', 'nisar_flood.cli',
                               *(str(a) for a in arguments)],
                              env=environment, text=True, capture_output=True)
    print(finished.stdout, end='')
    if finished.returncode:
        print(finished.stderr, file=sys.stderr)
    return finished.returncode


print('Colab' if IN_COLAB else 'local', '|', os.getcwd())
nisar('--help')


## 2. Say who you are and which chips are yours

`MY_CHIPS` are positions in the granule, written `row,col`. Varun assigns them so two
people do not label the same chip by accident — except in the calibration round (#29),
where everyone gets the *same* list deliberately.

**A position is the name.** Position `8800,13200` produces:

```
chip     N023_20260618_008800_013200.tif
geojson  N023_20260618_008800_013200.geojson
mask     N023_20260618_008800_013200_label.tif
```

So nobody types a filename, and two people given the same position get identical chips.


In [ ]:
LABELLER = 'your-github-handle'      # <-- change this

GRANULE_ID = 'N023_20260618'          # filename prefix for every chip
ACQUIRED   = '2026-06-18'             # acquisition date

MY_CHIPS = [
    '8800,13200',
    '7600,5600',
]

assert LABELLER != 'your-github-handle', 'put your own handle in LABELLER first'
print(f'{LABELLER}: {len(MY_CHIPS)} chips from {GRANULE_ID}')


## 3. Find the granule

**Locally** it looks in `data/raw/`. **In Colab** it reads the team's shared Drive
folder, `My Drive / nisar-flood / raw_granules`, so one copy serves everybody rather
than four people each downloading 1.8 GB.

That folder holds all four dual-pol granules, so the cell matches the one
`GRANULE_ID` names instead of guessing. To label a different date, change
`GRANULE_ID` and `ACQUIRED` in the cell above and nothing here.

| `GRANULE_ID` | `ACQUIRED` | regime |
|---|---|---|
| `N023_20260618` | `2026-06-18` | high water, near peak |
| `N025_20260712` | `2026-07-12` | high |
| `N029_20260829` | `2026-08-29` | falling |
| `N031_20260922` | `2026-09-22` | low |


In [ ]:
import glob

DRIVE_DIR = '/content/drive/MyDrive/nisar-flood/raw_granules'
LOCAL_DIR = 'data/raw'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    granule_dir = DRIVE_DIR
else:
    granule_dir = LOCAL_DIR

# Several granules sit in that folder, so match the one GRANULE_ID names
# rather than whichever happens to sort first.
orbit, day = GRANULE_ID.lstrip('N').split('_')
pattern = os.path.join(granule_dir, f'*GCOV_{orbit}_*DHDH*_{day}T*.h5')

found = sorted(glob.glob(pattern))
if len(found) != 1:
    listing = '\n'.join('   ' + os.path.basename(p)
                        for p in sorted(glob.glob(f'{granule_dir}/*.h5')))
    raise SystemExit(
        f'expected one granule matching {pattern}, found {len(found)}.\n'
        f'the folder holds:\n{listing or "   (nothing)"}')

GRANULE = found[0]
print(f'{os.path.basename(GRANULE)}\n{os.path.getsize(GRANULE)/1e9:.2f} GB')


In [ ]:
# Alternative: download it yourself from NASA Earthdata.
# Needs a free account at https://urs.earthdata.nasa.gov/users/new
# You type your own password; it is never stored in the notebook or the repo.
#
# !pip install -q earthaccess
# import earthaccess
# earthaccess.login()
# hits = earthaccess.search_data(short_name='NISAR_L2_GCOV_V1',
#                                granule_name='*023_068_D_090*DHDH*20260618*')
# GRANULE = earthaccess.download(hits[:1], 'data/raw')[0]


## 4. Cut your chips

`--only` is what makes this divisible: it reads just your windows out of the HDF5 rather
than converting the whole 1.8 GB granule, so this takes seconds instead of five minutes.

A chip is 512×512 at 20 m, so 10.24 km across, with two bands — HH and HV, gamma-nought
in decibels. Positions where fewer than 70% of pixels are usable are skipped, so getting
back fewer chips than you asked for is normal, not an error.


In [ ]:
nisar('make-chips',
      '--granule', GRANULE,
      '--granule-id', GRANULE_ID,
      '--date', ACQUIRED,
      '--out', 'data/chips',
      '--only', *MY_CHIPS)


## 5. Create your annotation files

One `.geojson` per chip, named to match. Each holds a single placeholder polygon with an
empty `class` — it exists only so QGIS recognises the file as a polygon layer and lets
you draw into it. The export ignores it. Leave it there and do not give it a class.

The command will not overwrite an annotation file that already exists, so re-running this
cannot destroy work you have done.


In [ ]:
nisar('label-init',
      '--chips', 'data/chips',
      '--out', 'data/labels/annotations',
      '--labeller', LABELLER)

print()
for path in sorted(glob.glob('data/labels/annotations/*.geojson')):
    print(path)


## 6. Look at what you are about to label

Water is dark in both bands; forest is bright and speckled. The `-25` to `+5` dB stretch
is the same one you will set in QGIS, so this is what you will see there.


In [ ]:
import matplotlib.pyplot as plt, rasterio

paths = sorted(glob.glob('data/chips/*.tif'))
figure, axes = plt.subplots(len(paths), 2, figsize=(9, 4.6*len(paths)), squeeze=False)
for row, path in enumerate(paths):
    with rasterio.open(path) as chip:
        for column, band in enumerate((1, 2)):
            axes[row][column].imshow(chip.read(band), cmap='gray', vmin=-25, vmax=5)
            axes[row][column].set_title(
                f'{chip.descriptions[band-1]}\n{os.path.basename(path)}', fontsize=8)
            axes[row][column].axis('off')
plt.tight_layout(); plt.show()


## 7. Get the files to QGIS

In Colab this downloads a zip. Locally the files are already where you need them and the
cell just tells you where.


In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files
    archive = shutil.make_archive(f'/content/{LABELLER}_to_label', 'zip', 'data')
    files.download(archive)
else:
    print('Open these in QGIS:')
    for path in sorted(glob.glob('data/chips/*.tif')):
        print('   ', os.path.abspath(path))
    for path in sorted(glob.glob('data/labels/annotations/*.geojson')):
        print('   ', os.path.abspath(path))


### In QGIS

1. **Install QGIS** if you have not: <https://qgis.org/download/>. Free, all platforms.
2. **Drag the `.tif` in.** Band 1 is HH, band 2 is HV.
3. **Make water visible.** Layer Properties → Symbology → *Singleband gray*, band 1,
   Min `-25`, Max `5`. Water goes near-black, forest mid-grey. Check anything ambiguous
   against band 2 — real water is dark in both.
4. **Drag the matching `.geojson` in.**
5. **Click the pencil** (Toggle Editing), then **Add Polygon Feature**. Draw, right-click
   to close. A form opens.
6. **Fill the form:**

   | Field | What to put |
   |---|---|
   | `class` | `1` water, `2` suspected flooded vegetation, `-1` unusable or unsure |
   | `labeller` | your GitHub handle |
   | `labelled_on` | today, `YYYY-MM-DD` |
   | `evidence_tier` | `1` backscatter, `2` HAND + gauge, `3` reference layer |
   | `notes` | why — required for anything you marked `2` or `-1` |

7. **Save** (Ctrl+S), then click the pencil again to stop editing.

**Never draw class `0`.** Anything you leave undrawn becomes non-water automatically.
That is most of the chip, and drawing it would waste hours.

**Read `docs/labelling_protocol.md` before deciding anything.** Short version: dark and
low-lying is water; vegetation-textured but low-lying during high water is `2`, not water;
dark but high ground is radar shadow, which is `-1`, not water. If two labelled classes
both seem defensible, use `-1` — an abstained pixel costs nothing, a wrong one corrupts
the IoU.

**Gauge context for this granule:** Manaus stage was **28.47 m on 18 Jun 2026** — 88th
percentile of the year, 12 days before the annual peak. The floodplain *is* flooded, so
expect plenty of class `2`.


## 8. Bring your finished annotations back

In Colab, upload the `.geojson` files you edited — nothing else, the chips are already
here. Locally there is nothing to do: QGIS saved them in place.


In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files
    os.makedirs('data/labels/annotations', exist_ok=True)
    for name in files.upload():
        shutil.move(name, f'data/labels/annotations/{name}')
        print('placed', name)
else:
    print('QGIS already saved them in data/labels/annotations/ - nothing to upload.')


## 9. Export to label masks

This burns your polygons onto the chip's exact pixel grid, applying water, then excluded,
then no-data — so where polygons overlap, the more cautious label wins. Pixels outside the
radar swath are forced to `-1` whatever you drew over them.

Output is one `int16` GeoTIFF per chip plus `labels_manifest.csv` counting every class.


In [ ]:
nisar('label-export',
      '--chips', 'data/chips',
      '--annotations', 'data/labels/annotations',
      '--out', 'data/labels/masks')


## 10. Check it before handing it in

Three things: the mask lines up with the chip, the class mix is believable, and you have
not accidentally labelled the whole chip as water.


In [ ]:
import numpy as np

NAMES = {1: 'water', 0: 'non-water', 2: 'excluded', -1: 'no-data'}
for mask_path in sorted(glob.glob('data/labels/masks/*_label.tif')):
    chip_path = os.path.join('data/chips',
                             os.path.basename(mask_path).replace('_label.tif', '.tif'))
    with rasterio.open(mask_path) as mask, rasterio.open(chip_path) as chip:
        labels = mask.read(1)
        aligned = mask.transform == chip.transform and mask.crs == chip.crs
    print(f'\n{os.path.basename(mask_path)}   grid matches chip: {aligned}')
    for value, count in zip(*np.unique(labels, return_counts=True)):
        print(f'   {value:>3}  {NAMES.get(value, "?"):12s} {count:7d} px  {count/labels.size:6.1%}')


In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap

colours = ListedColormap(['#bdbdbd', '#f4f4f4', '#1f6fb4', '#e08a1e'])   # -1, 0, 1, 2
levels = BoundaryNorm([-1.5, -0.5, 0.5, 1.5, 2.5], colours.N)
for mask_path in sorted(glob.glob('data/labels/masks/*_label.tif')):
    chip_path = os.path.join('data/chips',
                             os.path.basename(mask_path).replace('_label.tif', '.tif'))
    with rasterio.open(chip_path) as chip, rasterio.open(mask_path) as mask:
        figure, axes = plt.subplots(1, 2, figsize=(9, 4.6))
        axes[0].imshow(chip.read(1), cmap='gray', vmin=-25, vmax=5)
        axes[0].set_title('HH dB')
        axes[1].imshow(mask.read(1), cmap=colours, norm=levels)
        axes[1].set_title('your labels')
        for axis in axes: axis.axis('off')
        figure.suptitle(os.path.basename(mask_path), fontsize=9)
        plt.tight_layout(); plt.show()


## 11. Hand it back

Only the annotations go into git — chips and masks are too big and are gitignored on
purpose. Then tick your box on the issue and say which evidence tier you leaned on. That
note is what makes the calibration round (#29) useful rather than just a number.


In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files
    archive = shutil.make_archive(f'/content/{LABELLER}_labelled', 'zip', 'data/labels')
    files.download(archive)
else:
    print('Commit these:')
    for path in sorted(glob.glob('data/labels/annotations/*.geojson')):
        print('   ', path)


---
## Appendix — how chip positions were chosen

For whoever assigns the work, not for labellers. Scores every position on the stride-400
grid by how much open water it contains and keeps a usable mix. A chip that is 95% water
teaches nothing; one with no water teaches less.


In [ ]:
import h5py

SIZE, STRIDE, STEP = 512, 400, 4      # STEP decimates the scan to keep it quick

grid = h5py.File(GRANULE, 'r')['science/LSAR/GCOV/grids/frequencyA']
height, width = grid['HHHH'].shape
coarse, mask = grid['HHHH'][::STEP, ::STEP], grid['mask'][::STEP, ::STEP]
with np.errstate(divide='ignore', invalid='ignore'):
    decibels = 10 * np.log10(coarse)
decibels = np.where(np.isfinite(decibels), decibels, np.nan)

valid = mask == 1                     # 1 is usable; 0 is partial, 255 is off-swath
water = valid & (decibels < -15)      # open water is specular, so very dark

def integral(image):
    table = np.zeros((image.shape[0] + 1, image.shape[1] + 1), np.int64)
    table[1:, 1:] = image.astype(np.int64).cumsum(0).cumsum(1)
    return table

water_table, valid_table = integral(water), integral(valid)
side = SIZE // STEP

def total(table, row, column):
    return (table[row + side, column + side] - table[row, column + side]
            - table[row + side, column] + table[row, column])

def starts(length):
    positions = list(range(0, length - SIZE + 1, STRIDE))
    if positions[-1] != length - SIZE:
        positions.append(length - SIZE)
    return positions

candidates = []
for row in starts(height):
    for column in starts(width):
        r, c = row // STEP, column // STEP
        if r + side > water.shape[0] or c + side > water.shape[1]:
            continue
        if total(valid_table, r, c) / side**2 < 0.999:   # fully-valid chips only
            continue
        candidates.append((total(water_table, r, c) / side**2, row, column))

mixed = sorted((item for item in candidates if 0.25 <= item[0] <= 0.50),
               key=lambda item: abs(item[0] - 0.35))
print(f'{len(candidates)} fully-valid positions, {len(mixed)} with a 25-50% water mix\n')
for fraction, row, column in mixed[:12]:
    print(f"    '{row},{column}',    # water ~{fraction:.0%}")
